# Step 6b — +GAN Augmentation Ablation (Contribution C1)

**RESS 2025 — GAN-Conformal-RUL**

Standalone companion to `06_train_model.ipynb` (the baseline anchor). Retrains the **same**
BiLSTM + attention + RUL-head model on training sets augmented with synthetic near-failure
windows drawn from the frozen per-fold WGAN-GP generators (Step 4), and compares against the
un-augmented baseline.

| | |
|---|---|
| **Model** | identical to baseline (`ModelConfig`, unchanged) |
| **Only difference** | training set gains synthetic Stage-3 windows at `augment_ratio` |
| **Synthetic RUL** | borrowed from the nearest real Stage-3 window (kNN tie in feature space) |
| **Reported** | per-stage normalised RMSE — **Stage 3 (near-failure) is the C1 target** |

**Reading the result.** Stage 3 is only ~13% of windows, so a genuine near-failure gain barely
moves the *pooled* RMSE. The hypothesis is tested by the **near-failure column** moving off the
baseline anchor (0.274), not by the overall number (0.216). Because the generator is conditioned
on stage but not on RUL, the content↔label tie is weak (kNN, not RUL-conditioned generation), so
+GAN *alone* may yield a modest Stage-3 gain; the larger move is expected at +GAN+MT. A small
improvement here is still a positive C1 result.

## 1. Setup — clone repo, mount Drive

In [ ]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_DIR = '/content/drive/MyDrive/ress_checkpoints'
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import (ModelConfig, RULTrainer, rul_metrics,
                   per_bearing_rmse, cumulative_rmse, phm_score)
from gan import StageGAN, GANConfig

### `augment_helper` import

Preferred: the helper lives in `src/augment_helper.py` (push it to the repo alongside the other
src files). The fallback cell below writes it locally if it isn't in the clone yet, so this
notebook runs before you've pushed. Once `src/augment_helper.py` is committed, the fallback is a
no-op and can be ignored.

In [ ]:
try:
    from augment_helper import build_augmented_training_set
    print('augment_helper imported from repo.')
except ModuleNotFoundError:
    print('augment_helper not in repo yet — writing a local copy.')
    _HELPER_SRC = r'''"""
augment_helper.py — Step 6b GAN-augmentation utilities (notebook-side)

Kept out of src/ deliberately: this is experiment-orchestration glue for the
+GAN ablation row, not a core library component. It calls the frozen
generators (gan.py) and the baseline trainer (model.py) without modifying
either.

Central function: build_augmented_training_set(). It takes one fold's real
training tensors and a loaded StageGAN, synthesises near-failure windows,
assigns each a RUL label by a nearest-neighbour tie to a real Stage-3 window,
and returns augmented (X, y_rul, y_stage) ready for RULTrainer.fit.

Why a nearest-neighbour tie rather than independent marginal sampling:
The generator is conditioned on STAGE only, not on RUL, so a synthetic window
carries no intrinsic RUL. Sampling a RUL independently from the real Stage-3
marginal keeps the label distribution honest but severs any link between a
window's content and its label — pure label noise on the RUL axis, which gives
+GAN little reason to sharpen regression. Borrowing the RUL of the nearest real
Stage-3 window (in scaled feature space) restores a weak but genuine
content->label correlation at ~no cost and no GAN change, so the near-failure
RUL *slope* is at least partially preserved in the synthetic set.
"""

import numpy as np
from scipy.spatial import cKDTree

# Match the clip applied to real windows in windowing.apply_scaler (log_clip).
# The generator has no output activation, so its samples are not bounded to
# this range; real training windows are. Clipping synthetic windows to the
# same support (a) puts them in the range the encoder saw at baseline and
# (b) makes the kNN distances below meaningful, since the tree is built from
# clipped real windows.
CLIP_SIGMA = 5.0


def _flatten(X):
    """(N, T, F) -> (N, T*F) for distance computation."""
    return np.asarray(X, dtype=np.float32).reshape(len(X), -1)


def synth_rul_by_knn(X_syn, X_real_s3, y_rul_real_s3):
    """Assign a RUL to each synthetic window by nearest real Stage-3 window.

    Args:
        X_syn:        (M, T, F) synthetic near-failure windows (already clipped)
        X_real_s3:    (R, T, F) real Stage-3 training windows
        y_rul_real_s3:(R,)      their RUL labels

    Returns:
        (M,) RUL labels, each borrowed from the synthetic window's nearest
        real Stage-3 neighbour in flattened feature space.
    """
    if len(X_real_s3) == 0:
        raise ValueError("no real Stage-3 windows to borrow RUL from")
    tree = cKDTree(_flatten(X_real_s3))
    _, idx = tree.query(_flatten(X_syn), k=1)
    return np.asarray(y_rul_real_s3, dtype=np.float32)[idx]


def build_augmented_training_set(gan, X_train, y_rul_train, y_stage_train,
                                 augment_ratio=1.0, target_stage=2,
                                 seed=42, verbose=True):
    """Augment one fold's training set with synthetic near-failure windows.

    Args:
        gan:            a loaded StageGAN (generator in eval-ready state)
        X_train:        (N, T, F) scaled real training windows
        y_rul_train:    (N,) real RUL labels in [0, 1]
        y_stage_train:  (N,) 0-indexed stage labels {0,1,2}
        augment_ratio:  synthetic count = ratio * (# real target-stage windows)
        target_stage:   0-indexed stage to synthesise (2 = near-failure / S3)
        seed:           RNG seed for the final shuffle (reproducible ablation)

    Returns:
        X_aug, y_rul_aug, y_stage_aug — real + synthetic, shuffled together by
        one shared permutation so the three arrays stay aligned. When
        augment_ratio == 0 the inputs are returned unchanged (as arrays).
    """
    X_train = np.asarray(X_train, dtype=np.float32)
    y_rul_train = np.asarray(y_rul_train, dtype=np.float32)
    y_stage_train = np.asarray(y_stage_train, dtype=np.int64)

    real_tgt = y_stage_train == target_stage
    n_real_tgt = int(real_tgt.sum())
    n_syn = int(round(augment_ratio * n_real_tgt))

    if n_syn == 0 or n_real_tgt == 0:
        if verbose:
            print(f"  augment: ratio={augment_ratio}, "
                  f"real S3={n_real_tgt} -> 0 synthetic (returning real only)")
        return X_train, y_rul_train, y_stage_train

    # 1. sample synthetic windows from the frozen generator
    X_syn = gan.sample(n_syn, target_stage)                     # (M, T, F)

    # 2. clip to the same support as the real (log_clip) windows
    X_syn = np.clip(X_syn, -CLIP_SIGMA, CLIP_SIGMA).astype(np.float32)

    # 3. borrow a RUL from the nearest real Stage-3 window
    y_rul_syn = synth_rul_by_knn(
        X_syn, X_train[real_tgt], y_rul_train[real_tgt])
    y_stage_syn = np.full(n_syn, target_stage, dtype=np.int64)

    # 4. concatenate real + synthetic
    X_aug = np.concatenate([X_train, X_syn], axis=0)
    y_rul_aug = np.concatenate([y_rul_train, y_rul_syn], axis=0)
    y_stage_aug = np.concatenate([y_stage_train, y_stage_syn], axis=0)

    # 5. one shared permutation keeps the three arrays aligned
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(X_aug))

    if verbose:
        print(f"  augment: ratio={augment_ratio}, real S3={n_real_tgt} "
              f"-> +{n_syn} synthetic  (train {len(X_train)} -> {len(X_aug)})")
        print(f"           synthetic RUL borrowed by kNN: "
              f"mean {y_rul_syn.mean():.3f}  "
              f"[{y_rul_syn.min():.3f}, {y_rul_syn.max():.3f}]  "
              f"| real S3 RUL mean {y_rul_train[real_tgt].mean():.3f}")

    return X_aug[perm], y_rul_aug[perm], y_stage_aug[perm]
'''
    with open(f'{REPO_PATH}/src/augment_helper.py', 'w') as fh:
        fh.write(_HELPER_SRC)
    from augment_helper import build_augmented_training_set
    print('local augment_helper written and imported.')

## 2. Rebuild data (Steps 2–3, log_clip scaling)

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'not found: {CANDIDATES}'

all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)

fold_data = prepare_all_folds(results, scaling_method='log_clip', verbose=False)
folds = build_folds(results)
print(f'{len(fold_data)} folds ready.')

## 3. Load the five frozen generators from Drive

Same loader as `04_train_gan.ipynb` (cell 8). Rebuilds each `StageGAN` from its checkpoint's saved `config`, then loads generator + critic weights. Only the generator is used here.

In [ ]:
gans = {}
for k in range(1, 6):
    path = f'{CKPT_DIR}/gan_fold{k}.pt'
    if not os.path.exists(path):
        print(f'  fold {k}: checkpoint MISSING at {path}')
        continue
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    cfg = GANConfig(**ckpt['config'])
    g = StageGAN(cfg)
    g.G.load_state_dict(ckpt['generator'])
    g.D.load_state_dict(ckpt['critic'])
    gans[k] = g
    wd = ckpt.get('history', {}).get('wasserstein', [float('nan')])
    print(f"  fold {k}: loaded  (final W-dist {wd[-1]:.2f})")
print(f'\n{len(gans)}/5 GANs loaded from {CKPT_DIR}')
assert len(gans) == 5, 'need all 5 generators before running the ablation'

## 4. Ablation knobs and helpers

In [ ]:
AUGMENT_RATIO = 1.0    # Step 9 ablation knob: 1.0 = double the real S3 count
TARGET_STAGE  = 2      # 0-indexed near-failure (S3)

# Identical to the baseline anchor. Do NOT change this here, or the +GAN vs
# baseline comparison is confounded by a model-capacity difference rather
# than isolating the effect of augmentation.
cfg_model = ModelConfig(epochs=150, patience=25, lr=5e-4)


def per_stage_rmse(y_true, y_pred, stage):
    """Normalised RMSE per 0-indexed stage, plus overall."""
    out = {}
    for s in (0, 1, 2):
        m = stage == s
        out[s] = float(np.sqrt(np.mean((y_pred[m] - y_true[m]) ** 2))) if m.sum() else float('nan')
    out['overall'] = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    return out

## 5. Run baseline vs +GAN across all folds

Both variants use the identical model config and the identical validation / test partitions.
The only thing that changes between them is whether synthetic near-failure windows are added to
the training set. Per fold, the `+GAN` run prints the kNN-borrowed RUL spread — **watch for
`min ≈ max`, which would signal that fold's generator is emitting near-duplicates** (mode
collapse) and its augmentation should not be trusted.

In [ ]:
def run_variant(augment):
    """Train all folds; return dict of per-stage RMSE lists. augment=False -> baseline."""
    stage_rmse = {0: [], 1: [], 2: [], 'overall': []}
    for d, f in zip(fold_data, folds):
        k = d['fold']
        Xtr, rtr, str_ = d['X_train'], d['y_rul_train'], d['y_stage_train']

        if augment:
            Xtr, rtr, str_ = build_augmented_training_set(
                gans[k], Xtr, rtr, str_,
                augment_ratio=AUGMENT_RATIO, target_stage=TARGET_STAGE,
                verbose=True)

        tr = RULTrainer(cfg_model)
        tr.fit(Xtr, rtr, d['X_val'], d['y_rul_val'],
               stage_train=str_, stage_val=d['y_stage_val'],
               mask_healthy=True, verbose=False)

        yp = tr.predict(d['X_test'])
        r = per_stage_rmse(d['y_rul_test'], yp, d['y_stage_test'])
        for key in stage_rmse:
            stage_rmse[key].append(r[key])
        print(f"  fold {k}: near-failure RMSE {r[2]:.4f} | overall {r['overall']:.4f}")
    return stage_rmse


print('═'*60, '\nBASELINE (real data only)\n' + '═'*60)
base = run_variant(augment=False)
print('\n' + '═'*60, '\n+GAN (augment_ratio=%.1f)\n' % AUGMENT_RATIO + '═'*60)
aug = run_variant(augment=True)

## 6. Results — the C1 test

In [ ]:
names = {0: 'healthy', 1: 'early', 2: 'near-failure', 'overall': 'overall'}
rows = []
for key in (0, 1, 2, 'overall'):
    b = np.nanmean(base[key]); a = np.nanmean(aug[key])
    rows.append({'stage': names[key], 'baseline': b, '+GAN': a, 'delta': a - b})
res = pd.DataFrame(rows)

print(f"\n{'stage':14s}{'baseline':>12s}{'+GAN':>12s}{'delta':>10s}")
print('─'*48)
for _, r in res.iterrows():
    flag = '  <- C1 target' if r['stage'] == 'near-failure' else ''
    print(f"{r['stage']:14s}{r['baseline']:>12.4f}{r['+GAN']:>12.4f}{r['delta']:>+10.4f}{flag}")
print('─'*48)
print('negative delta = augmentation reduced error (good)')
print('\nanchors: near-failure 0.274 | overall 0.216')

In [ ]:
# Per-fold near-failure detail — dispersion matters given the 7x test-size spread
print(f"{'fold':6s}{'base S3':>10s}{'+GAN S3':>10s}{'delta':>10s}")
print('─'*36)
for i, k in enumerate([d['fold'] for d in fold_data]):
    b = base[2][i]; a = aug[2][i]
    print(f"{k:<6d}{b:>10.4f}{a:>10.4f}{a-b:>+10.4f}")

In [ ]:
# Bar chart: per-stage baseline vs +GAN
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['healthy', 'early', 'near-failure', 'overall']
x = np.arange(len(labels)); w = 0.38
b_vals = [np.nanmean(base[k]) for k in (0,1,2,'overall')]
a_vals = [np.nanmean(aug[k])  for k in (0,1,2,'overall')]
ax.bar(x - w/2, b_vals, w, label='baseline', color='#8c8c8c')
ax.bar(x + w/2, a_vals, w, label='+GAN', color='#e34948')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('normalised RMSE'); ax.set_title(f'+GAN ablation (augment_ratio={AUGMENT_RATIO})')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06b_gan_ablation_per_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## Next

- If near-failure RMSE drops meaningfully, C1 is supported — record this as the **+GAN** row of the Step 9 ablation table.
- Then layer **+Multi-task** (stage-classification head, combined loss) — expected to compound the near-failure gain, since the shared encoder benefits from the same synthetic windows on the classification objective.
- Sweep `AUGMENT_RATIO` (0.5, 1.0, 2.0) once the direction is confirmed, for the ablation's augmentation-strength curve.
- Then Step 7: split conformal prediction on the calibration set — the payoff of a sharper near-failure predictor is narrower intervals at the same coverage.